# Proyecto ETL – Contaminación, Mortalidad y Población por Localidad (Bogotá)

Este notebook construye un mini data warehouse a partir de 4 fuentes de datos:

- **IBOCA** (calidad del aire por estación)
- **SISAIRE** (contaminantes por estación/localidad)
- **Medicina** (mortalidad por localidad)
- **Población** (población por localidad, año y sexo – `datospoblacion.csv`)

El objetivo final es generar **5 tablas** limpias y consistentes:

1. `dim_fecha`
2. `dim_localidad`
3. `dim_localidad_fecha`  
4. `fact_mortalidad`
5. `fact_momento`
6. `fact_tasas_mortalidad` (Localidad + Fecha con población total, hombres y mujeres)

En las siguientes celdas haremos:

0. Setup general (imports, rutas).
1. Carga de datos "raw".
2. Limpieza y normalización por fuente.
3. Agregación a nivel **mes–localidad**.
4. Construcción de dimensiones.
5. Construcción de tablas de hechos.
6. Export final de CSV.


In [135]:
# 0. SETUP GENERAL

import pandas as pd
from pathlib import Path
import re
import unicodedata
import difflib

# Rutas base (ajusta si tu estructura de carpetas es distinta)
RUTA_SISAIRE = Path("Datos/SISAIRE")
RUTA_IBOCA = Path("Datos/IBOCA")
RUTA_MEDICINA = Path("Datos/Medicina")
RUTA_POBLACION = Path("Datos/datospoblacion.csv")
RUTA_SALIDA = Path("Output_ETL")

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

# Mapa de número de mes a nombre de mes en español
MAPA_NOMBRE_MES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

# Lista de nombres "oficiales" de localidades, en formato simple
CANONICAL_LOCALIDADES = [
    "Bogota",
    "Usaquen",
    "Chapinero",
    "Santa Fe",
    "San Cristobal",
    "Usme",
    "Tunjuelito",
    "Bosa",
    "Kennedy",
    "Fontibon",
    "Engativa",
    "Suba",
    "Barrios Unidos",
    "Teusaquillo",
    "Los Martires",
    "Antonio Narino",
    "Puente Aranda",
    "La Candelaria",
    "Rafael Uribe Uribe",
    "Ciudad Bolivar",
    "Sumapaz",
]

CANONICAL_LOCALIDADES_L = [c.lower() for c in CANONICAL_LOCALIDADES]

def normalizar_localidad_texto(x):
    """
    Normaliza nombres de localidad de forma agresiva para que se unifiquen
    variantes como:
      - '19 - Ciudad BolÃ­var'
      - 'Ciudad Bolívar'
      - 'bolivar'
      - 'bolirar', 'bolibar', etc.
    Todo termina en 'Ciudad Bolivar', etc.

    También quita tildes, mayúsculas y prefijos numéricos.
    """
    if not isinstance(x, str):
        return x

    s = x.strip()

    # Intentar reparar errores típicos de latin-1/utf-8 (BolÃ­var → Bolívar)
    try:
        s2 = s.encode("latin-1").decode("utf-8")
        s = s2
    except Exception:
        pass

    # Quitar BOM raro si existe
    s = s.replace("\ufeff", "")

    # Quitar prefijo tipo '01 - ' o '19-'
    s = re.sub(r"^\d+\s*-\s*", "", s)

    # A minúsculas para procesar
    s = s.lower().strip()

    # Quitar palabras genéricas que no ayudan
    basura = ["localidad", "loc.", "ciudad de", "ciudad", "d.c.", "bogota", "bogotá"]
    for w in basura:
        s = s.replace(w, "")

    # Colapsar espacios
    s = re.sub(r"\s+", " ", s).strip()

    # Quitar tildes para comparar
    s_norm = unicodedata.normalize("NFKD", s)
    s_sin_tilde = "".join(c for c in s_norm if not unicodedata.combining(c))

    # ---------------- REGLAS ESPECIALES ---------------- #

    # Todo lo que huela a bolivar/bolirar → Ciudad Bolivar
    if ("boliv" in s_sin_tilde) or ("bolir" in s_sin_tilde) or ("bolvar" in s_sin_tilde):
        return "Ciudad Bolivar"

    # Podemos meter aquí más reglas hardcode si quieres:
    # if "usaqu" in s_sin_tilde: return "Usaquen"
    # if "martir" in s_sin_tilde: return "Los Martires"
    # etc.

    # --------------------------------------------------- #

    # Fuzzy matching contra lista de localidades canónicas
    # Ej: 'rafael uribe' → 'Rafael Uribe Uribe'
    match = None
    if s_sin_tilde:
        matches = difflib.get_close_matches(
            s_sin_tilde,
            [c.lower() for c in CANONICAL_LOCALIDADES],
            n=1,
            cutoff=0.7  # puedes bajar un poco el cutoff si quieres aún más agresivo
        )
        if matches:
            idx = CANONICAL_LOCALIDADES_L.index(matches[0])
            match = CANONICAL_LOCALIDADES[idx]

    if match:
        return match

    # Si no encontramos nada, devolvemos el texto "bonito":
    return s_sin_tilde.title()

print("Rutas configuradas. Carpeta de salida:", RUTA_SALIDA)


Rutas configuradas. Carpeta de salida: Output_ETL


## 1. Carga de datos "raw"

En esta sección solo **leemos** los archivos y los concatenamos, sin hacer limpiezas profundas.

- `raw_iboca`  ← todos los archivos de IBOCA.
- `raw_sisaire` ← todos los archivos de SISAIRE.
- `raw_medicina` ← archivo(s) de mortalidad.
- `raw_poblacion` ← archivo `datospoblacion.csv` con población por localidad, año y sexo.

Más adelante vamos a limpiarlos y normalizarlos.


In [104]:
# 1.1 CARGA IBOCA → raw_iboca

import os

def cargar_iboca(ruta_carpeta: Path) -> pd.DataFrame:
    """
    Recorre todos los archivos de IBOCA en la carpeta y arma un DataFrame unificado.
    Ajusta esta función según el formato real de tus archivos.
    """
    dfs = []

    for fichero in ruta_carpeta.iterdir():
        if not fichero.suffix.lower() in [".xlsx", ".xls", ".csv"]:
            continue

        print("Leyendo IBOCA:", fichero.name)

        if fichero.suffix.lower() in [".xlsx", ".xls"]:
            df_raw = pd.read_excel(fichero, header=None)
        else:
            df_raw = pd.read_csv(fichero, header=None)

        # TODO: ADAPTAR A TU FORMATO REAL
        # ------------------------------------------------------------------
        # Ejemplo genérico: supongamos que:
        # - La fila 4 (index 3 o 4) tiene nombres de estaciones.
        # - Luego hay bloques de 4 columnas: [FechaHora, Concentración, NowCast, IBOCA]
        # Este es solo un template, revísalo con tu estructura real.
        # ------------------------------------------------------------------

        # Buscar fila donde están los nombres de las estaciones
        # (ajusta el índice según tu archivo real)
        fila_estaciones_idx = 4
        fila_estaciones = df_raw.iloc[fila_estaciones_idx]

        # Primera columna se asume que es fecha/hora, el resto son bloques
        # Aquí asumimos que cada estación ocupa 3 columnas (Concentración, NowCast, IBOCA)
        # después de una columna de fecha/hora (esto es solo un ejemplo)
        fecha_col = df_raw.columns[0]

        for col_inicio in range(1, df_raw.shape[1], 3):
            cols = df_raw.columns[col_inicio:col_inicio+3]
            if len(cols) < 3:
                continue

            nombre_estacion = fila_estaciones[cols[0]]
            if pd.isna(nombre_estacion):
                continue

            bloque = df_raw.loc[fila_estaciones_idx+1:, [fecha_col] + list(cols)].copy()
            bloque.columns = ["FechaHora", "Concentracion", "NowCast", "IBOCA"]
            bloque["Location"] = str(nombre_estacion)

            dfs.append(bloque)

    if not dfs:
        print("⚠️ No se cargó ningún bloque de IBOCA. Revisa el formato o la ruta.")
        return pd.DataFrame(columns=["FechaHora", "Concentracion", "NowCast", "IBOCA", "Location"])

    df_iboca = pd.concat(dfs, ignore_index=True)

    print("raw_iboca cargado. Shape:", df_iboca.shape)
    return df_iboca


raw_iboca = cargar_iboca(RUTA_IBOCA)
raw_iboca.head()


Leyendo IBOCA: IBOCA-PM10-2021-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2023-3.xlsx
Leyendo IBOCA: IBOCA-PM10-2024-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2021-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2024-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2023-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2023-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2022-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2022-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2020-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2025-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2020-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2025-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2020-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2025-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2020-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2025-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2022-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2022-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2023-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2023-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2021-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2024-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2023-3.xlsx
Leyendo IBOCA: IBOCA-PM25-2021-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2024-2.xlsx
raw_iboca ca

,FechaHora,Concentracion,NowCast,IBOCA,Location
0,Fecha & Hora,PM2.5 µg/m3,NaN,NaN,Bolivia
1,NaN,Concentración,Media móvil,IBOCA,Bolivia
2,2021-01-01 00:00:00,32,31,90.58,Bolivia
3,2021-01-01 01:00:00,37,31.4,91.43,Bolivia
4,2021-01-01 02:00:00,62,33.1,95.07,Bolivia


In [105]:
# 1.2 CARGA SISAIRE → raw_sisaire

def cargar_sisaire(ruta_carpeta: Path) -> pd.DataFrame:
    """
    Lee todos los archivos de SISAIRE (csv/xlsx) y los concatena.
    Asume que todos tienen columnas compatibles.
    """
    dfs = []

    for fichero in ruta_carpeta.iterdir():
        if not fichero.suffix.lower() in [".xlsx", ".xls", ".csv"]:
            continue

        print("Leyendo SISAIRE:", fichero.name)

        if fichero.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(fichero)
        else:
            df = pd.read_csv(fichero)

        dfs.append(df)

    if not dfs:
        print("⚠️ No se cargó ningún archivo de SISAIRE.")
        return pd.DataFrame()

    df_sisaire = pd.concat(dfs, ignore_index=True)
    print("raw_sisaire cargado. Shape:", df_sisaire.shape)
    return df_sisaire


raw_sisaire = cargar_sisaire(RUTA_SISAIRE)
raw_sisaire.head()


Leyendo SISAIRE: SISAIRE-NO2-2025.csv
Leyendo SISAIRE: SISAIRE-NO2-2024.csv
Leyendo SISAIRE: SISAIRE-O3-2024.csv
Leyendo SISAIRE: SISAIRE-O3-2025.csv
Leyendo SISAIRE: SISAIRE-O3-2021.csv
Leyendo SISAIRE: SISAIRE-NO2-2023.csv
Leyendo SISAIRE: SISAIRE-NO2-2022.csv
Leyendo SISAIRE: SISAIRE-O3-2020.csv
Leyendo SISAIRE: SISAIRE-O3-2022.csv
Leyendo SISAIRE: SISAIRE-NO2-2020.csv
Leyendo SISAIRE: SISAIRE-NO2-2021.csv
Leyendo SISAIRE: SISAIRE-O3-2023.csv
Leyendo SISAIRE: SISAIRE-SO2-2023.csv
Leyendo SISAIRE: SISAIRE-CO-2023.csv
Leyendo SISAIRE: SISAIRE-CO-2022.csv
Leyendo SISAIRE: SISAIRE-SO2-2022.csv
Leyendo SISAIRE: SISAIRE-SO2-2020.csv
Leyendo SISAIRE: SISAIRE-CO-2020.csv
Leyendo SISAIRE: SISAIRE-CO-2021.csv
Leyendo SISAIRE: SISAIRE-SO2-2021.csv
Leyendo SISAIRE: SISAIRE-SO2-2025.csv
Leyendo SISAIRE: SISAIRE-CO-2025.csv
Leyendo SISAIRE: SISAIRE-CO-2024.csv
Leyendo SISAIRE: SISAIRE-SO2-2024.csv
raw_sisaire cargado. Shape: (2622273, 7)


,Estacion,Fecha inicial,Fecha final,NO2,O3,SO2,CO
0,"""USME""",2025-07-31 22:00,2025-07-31 22:59,39.30036,NaN,NaN,NaN
1,"""USME""",2025-07-31 21:00,2025-07-31 21:59,44.94156,NaN,NaN,NaN
2,"""USME""",2025-07-31 20:00,2025-07-31 20:59,44.00136,NaN,NaN,NaN
3,"""USME""",2025-07-31 19:00,2025-07-31 19:59,27.07776,NaN,NaN,NaN
4,"""USME""",2025-07-31 18:00,2025-07-31 18:59,11.09436,NaN,NaN,NaN


In [106]:
# 1.3 CARGA MEDICINA → raw_medicina (versión con encoding)

def cargar_medicina(ruta_carpeta: Path) -> pd.DataFrame:
    """
    Carga el/los archivos de mortalidad (Medicina).
    Intenta varios encodings típicos en datos con tildes/ñ.
    """
    dfs = []

    for fichero in ruta_carpeta.iterdir():
        if not fichero.suffix.lower() in [".csv", ".xlsx", ".xls"]:
            continue

        print("Leyendo Medicina:", fichero.name)

        if fichero.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(fichero)
        else:
            # Intentamos primero con ; y latin-1
            try:
                df = pd.read_csv(fichero, sep=";", encoding="latin-1")
            except UnicodeDecodeError:
                # Plan B: ISO-8859-1
                try:
                    df = pd.read_csv(fichero, sep=";", encoding="ISO-8859-1")
                except UnicodeDecodeError:
                    # Último recurso: sin separador explícito pero con latin-1
                    df = pd.read_csv(fichero, encoding="latin-1")

        dfs.append(df)

    if not dfs:
        print("⚠️ No se cargó ningún archivo de Medicina.")
        return pd.DataFrame()

    df_med = pd.concat(dfs, ignore_index=True)
    print("raw_medicina cargado. Shape:", df_med.shape)
    return df_med

raw_medicina = cargar_medicina(RUTA_MEDICINA)
raw_medicina.head()


Leyendo Medicina: datosmedicina.csv
raw_medicina cargado. Shape: (47542, 12)


,ANO,MES,EPS,SUBRED,SEXO,MIGRANTE,REGIMEN_SEGURIDAD_SOCIAL,EDAD_FALLECIDO,EDAD_QUINQUENAL,CIE10_AGRUPADA,CIE10_BASICA,LOCALIDAD
0,2015,2,ALIANSALUD E.P.S.,NORTE,FEMENINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
1,2015,4,OTROS,NORTE,FEMENINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
2,2021,1,E.P.S. SANITAS,NORTE,MASCULINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
3,2021,6,E.P.S. SANITAS,NORTE,MASCULINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
4,2022,2,FAMISANAR E.P.S. LTDA - CAFAM - COLSUBSIDIO,NORTE,MASCULINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá


In [107]:
# 1.4 CARGA POBLACIÓN → raw_poblacion (versión corregida)

# 1) Leer el archivo usando latin-1 porque los acentos vienen en ese encoding.
raw_poblacion = pd.read_csv(
    RUTA_POBLACION,
    sep=";", 
    encoding="latin-1"
)

# 2) Quitar el BOM de la primera columna si existe
raw_poblacion.columns = [
    col.replace("ï»¿", "") for col in raw_poblacion.columns
]

# 3) Arreglar textos mal decodificados: UsaquÃ©n → Usaquén
def fix_text(x):
    if isinstance(x, str):
        try:
            return x.encode("latin-1").decode("utf-8")
        except:
            return x
    return x

raw_poblacion = raw_poblacion.applymap(fix_text)

print("raw_poblacion cargado. Shape:", raw_poblacion.shape)
raw_poblacion.head()


/var/folders/jq/wqkcvg155zg7f2xm3sy80x5h0000gn/T/ipykernel_34884/4147251056.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  raw_poblacion = raw_poblacion.applymap(fix_text)


raw_poblacion cargado. Shape: (131502, 10)


,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,ORDEN_MCV,CURSODEVIDA,ORDEN_GRUPO_EDAD,GRUPOEDAD,POBLACION_7
0,2005,1,Usaquén,Hombres,0,1,Primera Infancia,1,0 a 4,2909
1,2005,1,Usaquén,Hombres,1,1,Primera Infancia,1,0 a 4,2954
2,2005,1,Usaquén,Hombres,2,1,Primera Infancia,1,0 a 4,2919
3,2005,1,Usaquén,Hombres,3,1,Primera Infancia,1,0 a 4,2989
4,2005,1,Usaquén,Hombres,4,1,Primera Infancia,1,0 a 4,3079


## 2. Limpieza y normalización

En esta sección transformamos cada fuente a una versión **clean** con las columnas
que vamos a usar después:

- `iboca_clean`   → FechaHora, Año, Mes, Localidad, Concentración, NowCast, IBOCA.
- `sisaire_clean` → Fecha, Año, Mes, Localidad, NO2, O3, SO2, CO.
- `medicina_clean`→ Año, Mes, Localidad, Fecha (primer día del mes), métricas de muertes.
- `poblacion_clean` → Año, Código_Localidad, Localidad, Poblacion_Total, Hombres, Mujeres.


In [108]:
# 2.1 LIMPIEZA IBOCA → iboca_clean (con normalización de Localidad)

def limpiar_iboca(raw_iboca: pd.DataFrame) -> pd.DataFrame:
    df = raw_iboca.copy()

    # FechaHora a datetime
    df["FechaHora"] = pd.to_datetime(df["FechaHora"], errors="coerce")

    # Año y Mes
    df["Ano"] = df["FechaHora"].dt.year
    df["Mes"] = df["FechaHora"].dt.month

    # Limpieza numérica básica
    for col in ["Concentracion", "NowCast", "IBOCA"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.replace("sin dato", "", case=False, regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Mapear Location → Localidad (AQUÍ VA TU DICCIONARIO)
    mapa_estacion_localidad = {
        # TODO: Rellena con tus mappings reales:
        # "NombreEstaciónEnIBOCA": "Ciudad Bolívar",
        # "KENNEDY": "Kennedy",
    }

    df["Localidad"] = df["Location"].map(mapa_estacion_localidad).fillna(df["Location"])

    # Normalizar texto de localidad (arregla BolÃ­var, etc.)
    df["Localidad"] = df["Localidad"].apply(normalizar_localidad_texto)

    # Nos quedamos con las columnas clave
    columnas = [
        "FechaHora", "Ano", "Mes", "Localidad",
        "Concentracion", "NowCast", "IBOCA"
    ]
    df = df[columnas].dropna(subset=["FechaHora"])

    print("iboca_clean listo. Shape:", df.shape)
    return df


iboca_clean = limpiar_iboca(raw_iboca)
iboca_clean.head()


/var/folders/jq/wqkcvg155zg7f2xm3sy80x5h0000gn/T/ipykernel_34884/3810684982.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FechaHora"] = pd.to_datetime(df["FechaHora"], errors="coerce")


iboca_clean listo. Shape: (1937443, 7)


,FechaHora,Ano,Mes,Localidad,Concentracion,NowCast,IBOCA
2,2021-01-01 00:00:00,2021.0,1.0,Ciudad Bolivar,32.0,31.0,90.58
3,2021-01-01 01:00:00,2021.0,1.0,Ciudad Bolivar,37.0,31.4,91.43
4,2021-01-01 02:00:00,2021.0,1.0,Ciudad Bolivar,62.0,33.1,95.07
5,2021-01-01 03:00:00,2021.0,1.0,Ciudad Bolivar,62.0,34.7,98.50
6,2021-01-01 04:00:00,2021.0,1.0,Ciudad Bolivar,94.0,37.9,106.12


In [109]:
# 2.2 LIMPIEZA SISAIRE → sisaire_clean (usando 'Fecha final' y 'Estacion')

def limpiar_sisaire(raw_sisaire: pd.DataFrame) -> pd.DataFrame:
    df = raw_sisaire.copy()

    print("Columnas originales de SISAIRE:")
    print(df.columns.tolist())

    # 1) Renombrar columnas clave a nombres estándar
    df = df.rename(
        columns={
            "Fecha final": "Fecha",
            "Estacion": "Localidad"
        }
    )

    # 2) Convertir Fecha a datetime
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

    # 3) Año y mes
    df["Ano"] = df["Fecha"].dt.year
    df["Mes"] = df["Fecha"].dt.month

    # 4) Normalizar texto de localidad
    df["Localidad"] = df["Localidad"].apply(normalizar_localidad_texto)

    # 5) Limpiar contaminantes numéricos
    contaminantes = ["NO2", "O3", "SO2", "CO"]

    for col in contaminantes:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", ".", regex=False)
                .str.replace("sin dato", "", case=False, regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    cols_finales = ["Fecha", "Ano", "Mes", "Localidad"] + [
        c for c in contaminantes if c in df.columns
    ]
    df = df[cols_finales].dropna(subset=["Fecha"])

    print("sisaire_clean listo. Shape:", df.shape)
    return df


sisaire_clean = limpiar_sisaire(raw_sisaire)
sisaire_clean.head()


Columnas originales de SISAIRE:
['Estacion', 'Fecha inicial', 'Fecha final', 'NO2', 'O3', 'SO2', 'CO']
sisaire_clean listo. Shape: (2622273, 8)


,Fecha,Ano,Mes,Localidad,NO2,O3,SO2,CO
0,2025-07-31 22:59:00,2025,7,Usme,39.30036,NaN,NaN,NaN
1,2025-07-31 21:59:00,2025,7,Usme,44.94156,NaN,NaN,NaN
2,2025-07-31 20:59:00,2025,7,Usme,44.00136,NaN,NaN,NaN
3,2025-07-31 19:59:00,2025,7,Usme,27.07776,NaN,NaN,NaN
4,2025-07-31 18:59:00,2025,7,Usme,11.09436,NaN,NaN,NaN


In [110]:
# 2.3 LIMPIEZA MEDICINA → medicina_clean (versión final correcta)

def limpiar_medicina(raw_medicina: pd.DataFrame) -> pd.DataFrame:
    df = raw_medicina.copy()

    # --- 1. Renombrar columnas estándar ---
    df.rename(columns={
        "ANO": "Ano",
        "MES": "Mes",
        "LOCALIDAD": "Localidad",
        "SEXO": "Sexo",
        "CIE10_AGRUPADA": "CIE10_Agrupada",
        "CIE10_BASICA": "CIE10_Basica",
        "REGIMEN_SEGURIDAD_SOCIAL": "Regimen",
    }, inplace=True)

    # --- 2. Normalizar localidad ---
    df["Localidad"] = df["Localidad"].apply(normalizar_localidad_texto)

    # --- 3. Crear fecha YYYY-MM-01 ---
    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    # --- 4. Crear columnas de conteo ---
    # Cada fila = 1 muerte
    df["Muertes_Totales"] = 1

    # Por sexo
    sexo = df["Sexo"].astype(str).str.strip().str.upper()
    df["Muertes_Totales_Hombres"] = (sexo.str.startswith("M")).astype(int)
    df["Muertes_Totales_Mujeres"] = (sexo.str.startswith("F")).astype(int)

    # Por CIE10 agrupada y básica
    df["Muertes_CIE10_Agrupada"] = 1
    df["Muertes_CIE10_Basica"] = 1

    # --- 5. Agrupar por año, mes, localidad ---
    agg_cols = {
        "Muertes_Totales": "sum",
        "Muertes_Totales_Hombres": "sum",
        "Muertes_Totales_Mujeres": "sum",
        "Muertes_CIE10_Agrupada": "sum",
        "Muertes_CIE10_Basica": "sum",
    }

    df_group = (
        df
        .groupby(["Ano", "Mes", "Localidad", "Fecha"], as_index=False)
        .agg(agg_cols)
    )

    print("medicina_clean lista. Shape:", df_group.shape)
    return df_group


# Ejecutar
medicina_clean = limpiar_medicina(raw_medicina)
medicina_clean.head()


medicina_clean lista. Shape: (2482, 9)


,Ano,Mes,Localidad,Fecha,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Muertes_CIE10_Agrupada,Muertes_CIE10_Basica
0,2015,1,,2015-01-01,188,112,76,188,188
1,2015,1,Antonio Narino,2015-01-01,2,1,1,2,2
2,2015,1,Barrios Unidos,2015-01-01,6,3,3,6,6
3,2015,1,Bosa,2015-01-01,20,14,6,20,20
4,2015,1,Chapinero,2015-01-01,4,3,1,4,4


In [111]:
# 2.4 LIMPIEZA POBLACIÓN → poblacion_clean (normalizando Localidad)

def limpiar_poblacion(raw_poblacion: pd.DataFrame) -> pd.DataFrame:
    df = raw_poblacion.copy()

    # Asegurarnos de que POBLACION_7 sea numérico
    df["POBLACION_7"] = pd.to_numeric(df["POBLACION_7"], errors="coerce").fillna(0)

    df_agg = (
        df
        .groupby(["ANO", "CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD", "SEXO"], as_index=False)
        .agg({"POBLACION_7": "sum"})
    )

    df_pivot = df_agg.pivot_table(
        index=["ANO", "CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD"],
        columns="SEXO",
        values="POBLACION_7",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    df_pivot.columns.name = None

    col_hombres = "Hombres" if "Hombres" in df_pivot.columns else "HOMBRES"
    col_mujeres = "Mujeres" if "Mujeres" in df_pivot.columns else "MUJERES"

    df_pivot["Poblacion_Hombres"] = df_pivot.get(col_hombres, 0)
    df_pivot["Poblacion_Mujeres"] = df_pivot.get(col_mujeres, 0)
    df_pivot["Poblacion_Total"] = df_pivot["Poblacion_Hombres"] + df_pivot["Poblacion_Mujeres"]

    df_pivot.rename(
        columns={
            "ANO": "Ano",
            "CODIGO_LOCALIDAD": "Codigo_Localidad",
            "NOMBRE_LOCALIDAD": "Localidad",
        },
        inplace=True,
    )

    # Normalizar texto de localidad aquí también
    df_pivot["Localidad"] = df_pivot["Localidad"].apply(normalizar_localidad_texto)

    columnas_finales = [
        "Ano", "Codigo_Localidad", "Localidad",
        "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres"
    ]
    df_final = df_pivot[columnas_finales].copy()

    print("poblacion_clean lista. Shape:", df_final.shape)
    return df_final


poblacion_clean = limpiar_poblacion(raw_poblacion)
poblacion_clean.head()


poblacion_clean lista. Shape: (651, 6)


,Ano,Codigo_Localidad,Localidad,Poblacion_Total,Poblacion_Hombres,Poblacion_Mujeres
0,2005,0,,6710910,3236477,3474433
1,2005,1,Usaquen,415099,191068,224031
2,2005,2,Chapinero,121307,55515,65792
3,2005,3,Santa Fe,96046,48127,47919
4,2005,4,San Cristobal,400187,195556,204631


## 3. Agregación a nivel mes–localidad

En esta sección llevamos cada fuente a un grano común:

- **IBOCA** → promedio mensual por localidad (`iboca_mes_loc`)
- **SISAIRE** → promedio mensual por localidad (`sisaire_mes_loc`)
- **Medicina** → total mensual por localidad (`medicina_mes_loc`)

Todas estas tablas tendrán al menos: `Ano`, `Mes`, `Localidad`, `Fecha` (primer día del mes).


In [112]:
# 3.1 IBOCA → iboca_mes_loc (promedio mes–localidad)

def agregar_iboca_mensual(iboca_clean: pd.DataFrame) -> pd.DataFrame:
    if iboca_clean.empty:
        print("⚠️ iboca_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame(columns=[
            "Ano", "Mes", "Localidad", "Fecha",
            "Concentracion_Promedio", "NowCast_Promedio", "IBOCA_Promedio"
        ])

    df = (
        iboca_clean
        .groupby(["Ano", "Mes", "Localidad"], as_index=False)
        .agg({
            "Concentracion": "mean",
            "NowCast": "mean",
            "IBOCA": "mean"
        })
    )

    # Fecha = primer día del mes correspondiente
    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    df.rename(
        columns={
            "Concentracion": "Concentracion_Promedio",
            "NowCast": "NowCast_Promedio",
            "IBOCA": "IBOCA_Promedio",
        },
        inplace=True
    )

    # Reordenar columnas
    cols = ["Ano", "Mes", "Localidad", "Fecha",
            "Concentracion_Promedio", "NowCast_Promedio", "IBOCA_Promedio"]
    df = df[cols]

    print("iboca_mes_loc listo. Shape:", df.shape)
    return df


iboca_mes_loc = agregar_iboca_mensual(iboca_clean)
iboca_mes_loc.head()


iboca_mes_loc listo. Shape: (1293, 7)


,Ano,Mes,Localidad,Fecha,Concentracion_Promedio,NowCast_Promedio,IBOCA_Promedio
0,2020.0,1.0,Carvajal - Sevillana,2020-01-01,29.820690,29.539651,87.655780
1,2020.0,1.0,Cdar,2020-01-01,13.808123,13.576747,49.557460
2,2020.0,1.0,Fontibon,2020-01-01,19.038043,19.359677,65.257769
3,2020.0,1.0,Guaymaral,2020-01-01,15.274105,14.988441,54.107608
4,2020.0,1.0,Kennedy,2020-01-01,23.707713,23.670296,74.957742


In [113]:
# 3.2 SISAIRE → sisaire_mes_loc (promedio mes–localidad)

def agregar_sisaire_mensual(sisaire_clean: pd.DataFrame) -> pd.DataFrame:
    if sisaire_clean.empty:
        print("⚠️ sisaire_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame(columns=[
            "Ano", "Mes", "Localidad", "Fecha",
            "NO2_Promedio", "O3_Promedio", "SO2_Promedio", "CO_Promedio"
        ])

    contaminantes = [c for c in ["NO2", "O3", "SO2", "CO"] if c in sisaire_clean.columns]

    df = (
        sisaire_clean
        .groupby(["Ano", "Mes", "Localidad"], as_index=False)
        .agg({c: "mean" for c in contaminantes})
    )

    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    # Renombrar a *_Promedio
    rename_dict = {c: f"{c}_Promedio" for c in contaminantes}
    df.rename(columns=rename_dict, inplace=True)

    cols = ["Ano", "Mes", "Localidad", "Fecha"] + list(rename_dict.values())
    df = df[cols]

    print("sisaire_mes_loc listo. Shape:", df.shape)
    return df


sisaire_mes_loc = agregar_sisaire_mensual(sisaire_clean)
sisaire_mes_loc.head()


sisaire_mes_loc listo. Shape: (1084, 8)


,Ano,Mes,Localidad,Fecha,NO2_Promedio,O3_Promedio,SO2_Promedio,CO_Promedio
0,2020,1,Carvajal - Sevillana,2020-01-01,44.197494,21.311795,8.618221,1110.075958
1,2020,1,Centro De Alto Rendimiento,2020-01-01,26.208586,34.033379,3.198179,912.327544
2,2020,1,Guaymaral,2020-01-01,21.656340,26.511976,NaN,NaN
3,2020,1,Kennedy,2020-01-01,33.802356,40.905034,4.382786,780.147598
4,2020,1,Las Ferias,2020-01-01,30.938985,22.892488,NaN,767.146783


In [114]:
# 3.3 MEDICINA → medicina_mes_loc (total mes–localidad)

def agregar_medicina_mensual(medicina_clean: pd.DataFrame) -> pd.DataFrame:
    if medicina_clean.empty:
        print("⚠️ medicina_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame()

    cols_muertes = [
        c for c in medicina_clean.columns
        if c.startswith("Muertes_")
    ]

    agg_dict = {c: "sum" for c in cols_muertes}

    df = (
        medicina_clean
        .groupby(["Ano", "Mes", "Localidad", "Fecha"], as_index=False)
        .agg(agg_dict)
    )

    print("medicina_mes_loc listo. Shape:", df.shape)
    return df


medicina_mes_loc = agregar_medicina_mensual(medicina_clean)
medicina_mes_loc.head()


medicina_mes_loc listo. Shape: (2482, 9)


,Ano,Mes,Localidad,Fecha,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Muertes_CIE10_Agrupada,Muertes_CIE10_Basica
0,2015,1,,2015-01-01,188,112,76,188,188
1,2015,1,Antonio Narino,2015-01-01,2,1,1,2,2
2,2015,1,Barrios Unidos,2015-01-01,6,3,3,6,6
3,2015,1,Bosa,2015-01-01,20,14,6,20,20
4,2015,1,Chapinero,2015-01-01,4,3,1,4,4


## 4. Construcción de dimensiones

A partir de las tablas mensuales por localidad, vamos a construir:

1. `dim_fecha`       → catálogo de meses (PK_Fecha, Año, Mes, NombreMes)
2. `dim_localidad`   → catálogo de localidades (PK_Localidad, Localidad, Codigo_Localidad)
3. `dim_localidad_fecha` → combinación Localidad + Fecha con población total, hombres y mujeres


In [115]:
# 4.1 DIM_FECHA  (años válidos = años con datos REALES en las 4 bases)

def construir_dim_fecha(
    iboca_mes_loc: pd.DataFrame,
    sisaire_mes_loc: pd.DataFrame,
    medicina_mes_loc: pd.DataFrame,
    poblacion_clean: pd.DataFrame,
):
    """
    Construye dim_fecha y calcula los AÑOS VÁLIDOS como:
    - Años donde IBOCA tiene algún valor no nulo en sus columnas de promedio.
    - Años donde SISAIRE tiene algún contaminante promedio no nulo.
    - Años donde MEDICINA tiene Muertes_Totales > 0.
    - Años presentes en POBLACIÓN.

    Solo esos años se usan para dim_fecha.
    """

    # --- 1. Años con datos útiles en cada base ---

    anos_iboca = set()
    if not iboca_mes_loc.empty:
        cols_ibo = [c for c in ["Concentracion_Promedio",
                                "NowCast_Promedio",
                                "IBOCA_Promedio"] if c in iboca_mes_loc.columns]
        if cols_ibo:
            mask_ibo = iboca_mes_loc[cols_ibo].notna().any(axis=1)
            anos_iboca = set(iboca_mes_loc.loc[mask_ibo, "Ano"].unique())

    anos_sis = set()
    if not sisaire_mes_loc.empty:
        cols_sis = [c for c in ["NO2_Promedio",
                                "O3_Promedio",
                                "SO2_Promedio",
                                "CO_Promedio"] if c in sisaire_mes_loc.columns]
        if cols_sis:
            mask_sis = sisaire_mes_loc[cols_sis].notna().any(axis=1)
            anos_sis = set(sisaire_mes_loc.loc[mask_sis, "Ano"].unique())

    anos_med = set()
    if not medicina_mes_loc.empty and "Muertes_Totales" in medicina_mes_loc.columns:
        mask_med = medicina_mes_loc["Muertes_Totales"] > 0
        anos_med = set(medicina_mes_loc.loc[mask_med, "Ano"].unique())

    anos_pob = set()
    if not poblacion_clean.empty:
        anos_pob = set(poblacion_clean["Ano"].dropna().unique())

    sets_anos = [anos_iboca, anos_sis, anos_med, anos_pob]
    sets_anos = [s for s in sets_anos if s]  # quitar vacíos por seguridad

    if len(sets_anos) < 2:
        print("⚠️ No hay suficientes fuentes para cruzar años.")
        return pd.DataFrame(columns=["PK_Fecha", "Ano", "Mes", "NombreMes"]), []

    # --- 2. Intersección de años con datos reales en TODAS las bases ---
    anos_validos = sorted(set.intersection(*sets_anos))
    print("Años presentes con datos REALES en TODAS las bases:", anos_validos)

    if not anos_validos:
        print("⚠️ La intersección de años está vacía.")
        return pd.DataFrame(columns=["PK_Fecha", "Ano", "Mes", "NombreMes"]), []

    # --- 3. Construir dim_fecha solo con esos años ---

    frames = []
    for df in [iboca_mes_loc, sisaire_mes_loc, medicina_mes_loc]:
        if not df.empty:
            frames.append(df[["Ano", "Mes"]])

    if not frames:
        print("⚠️ No hay datos mensuales para construir dim_fecha.")
        return pd.DataFrame(columns=["PK_Fecha", "Ano", "Mes", "NombreMes"]), anos_validos

    fechas = (
        pd.concat(frames, ignore_index=True)
        .dropna()
        .drop_duplicates()
    )

    # Filtrar a los años válidos
    fechas = fechas[fechas["Ano"].isin(anos_validos)]
    fechas["Ano"] = fechas["Ano"].astype(int)
    fechas["Mes"] = fechas["Mes"].astype(int)

    fechas["PK_Fecha"] = fechas["Ano"] * 100 + fechas["Mes"]
    fechas["NombreMes"] = fechas["Mes"].map(MAPA_NOMBRE_MES)

    fechas = fechas.sort_values(["Ano", "Mes"]).reset_index(drop=True)
    fechas = fechas[["PK_Fecha", "Ano", "Mes", "NombreMes"]]

    print("dim_fecha lista. Shape:", fechas.shape)
    return fechas, anos_validos


# 👉 Ejecutar de nuevo
dim_fecha, ANOS_VALIDOS = construir_dim_fecha(
    iboca_mes_loc, sisaire_mes_loc, medicina_mes_loc, poblacion_clean
)

dim_fecha.head()


Años presentes con datos REALES en TODAS las bases: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
dim_fecha lista. Shape: (72, 4)


,PK_Fecha,Ano,Mes,NombreMes
0,202001,2020,1,Enero
1,202002,2020,2,Febrero
2,202003,2020,3,Marzo
3,202004,2020,4,Abril
4,202005,2020,5,Mayo


In [116]:
# 4.1bis Filtrar todas las tablas a los AÑOS VÁLIDOS

if ANOS_VALIDOS:
    iboca_mes_loc = iboca_mes_loc[iboca_mes_loc["Ano"].isin(ANOS_VALIDOS)].copy()
    sisaire_mes_loc = sisaire_mes_loc[ sisaire_mes_loc["Ano"].isin(ANOS_VALIDOS) ].copy()
    medicina_mes_loc = medicina_mes_loc[ medicina_mes_loc["Ano"].isin(ANOS_VALIDOS) ].copy()
    poblacion_clean = poblacion_clean[ poblacion_clean["Ano"].isin(ANOS_VALIDOS) ].copy()

    print("iboca_mes_loc años:", sorted(iboca_mes_loc["Ano"].unique()))
    print("sisaire_mes_loc años:", sorted(sisaire_mes_loc["Ano"].unique()))
    print("medicina_mes_loc años:", sorted(medicina_mes_loc["Ano"].unique()))
    print("poblacion_clean años:", sorted(poblacion_clean["Ano"].unique()))
else:
    print("⚠️ ANOS_VALIDOS está vacío. No se filtraron las tablas.")


iboca_mes_loc años: [np.float64(2020.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0), np.float64(2025.0)]
sisaire_mes_loc años: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
medicina_mes_loc años: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
poblacion_clean años: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [117]:
# 4.2 DIM_LOCALIDAD (solo localidades que aparecen en TODAS las bases)

def construir_dim_localidad(
    iboca_mes_loc: pd.DataFrame,
    sisaire_mes_loc: pd.DataFrame,
    medicina_mes_loc: pd.DataFrame,
    poblacion_clean: pd.DataFrame,
) -> pd.DataFrame:
    # Conjuntos de localidades en cada fuente mensual
    sets_locs = []

    if not iboca_mes_loc.empty:
        sets_locs.append(set(iboca_mes_loc["Localidad"].dropna().unique()))
    if not sisaire_mes_loc.empty:
        sets_locs.append(set(sisaire_mes_loc["Localidad"].dropna().unique()))
    if not medicina_mes_loc.empty:
        sets_locs.append(set(medicina_mes_loc["Localidad"].dropna().unique()))
    if not poblacion_clean.empty:
        sets_locs.append(set(poblacion_clean["Localidad"].dropna().unique()))

    if len(sets_locs) < 2:
        print("⚠️ No hay suficientes fuentes para cruzar localidades.")
        return pd.DataFrame(columns=["PK_Localidad", "Localidad", "Codigo_Localidad"])

    # INTERSECCIÓN de localidades en TODAS las fuentes
    locs_intersection = set.intersection(*sets_locs)

    print("Número de localidades en la intersección de todas las bases:", len(locs_intersection))

    base_locs = pd.DataFrame(
        {"Localidad": sorted(locs_intersection)}
    )

    # Traer Código de localidad desde población
    locs_pob = (
        poblacion_clean[["Codigo_Localidad", "Localidad"]]
        .drop_duplicates(subset=["Localidad"])
    )

    dim_loc = base_locs.merge(locs_pob, on="Localidad", how="left")

    # Crear PK_Localidad
    dim_loc = dim_loc.sort_values("Localidad").reset_index(drop=True)
    dim_loc.insert(0, "PK_Localidad", dim_loc.index + 1)

    print("dim_localidad lista. Shape:", dim_loc.shape)
    return dim_loc


dim_localidad = construir_dim_localidad(
    iboca_mes_loc, sisaire_mes_loc, medicina_mes_loc, poblacion_clean
)
dim_localidad.head()


Número de localidades en la intersección de todas las bases: 8
dim_localidad lista. Shape: (8, 3)


,PK_Localidad,Localidad,Codigo_Localidad
0,1,Ciudad Bolivar,19
1,2,Fontibon,9
2,3,Kennedy,8
3,4,Puente Aranda,16
4,5,San Cristobal,4


In [118]:
# 4.3 DIM_LOCALIDAD_FECHA (redefinida por si acaso)

def construir_dim_localidad_fecha(
    dim_fecha: pd.DataFrame,
    dim_localidad: pd.DataFrame,
    poblacion_clean: pd.DataFrame,
) -> pd.DataFrame:
    if dim_fecha.empty or dim_localidad.empty:
        print("⚠️ dim_fecha o dim_localidad están vacías. Devuelvo DataFrame vacío.")
        return pd.DataFrame()

    # Base: todas las combinaciones de Fecha (PK_Fecha, Ano, Mes) x Localidad
    base = (
        dim_fecha[["PK_Fecha", "Ano", "Mes"]]
        .assign(key=1)
        .merge(
            dim_localidad[["PK_Localidad", "Localidad"]].assign(key=1),
            on="key", how="outer"
        )
        .drop(columns="key")
    )

    # Unimos con población por Año y Localidad
    # poblacion_clean: Ano, Codigo_Localidad, Localidad, Poblacion_*
    pob = poblacion_clean.copy()

    df = base.merge(
        pob,
        on=["Ano", "Localidad"],
        how="left",
        suffixes=("", "_pob")
    )

    # Crear PK_LocalidadFecha
    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)
    df.insert(0, "PK_LocalidadFecha", df.index + 1)

    columnas = [
        "PK_LocalidadFecha",
        "PK_Fecha", "PK_Localidad",
        "Ano", "Mes", "Localidad",
        "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres"
    ]
    # Por si alguna columna de población no existe:
    columnas = [c for c in columnas if c in df.columns]

    df = df[columnas]

    print("dim_localidad_fecha lista. Shape:", df.shape)
    return df

# 👇 MUY IMPORTANTE: crear realmente la tabla
dim_localidad_fecha = construir_dim_localidad_fecha(
    dim_fecha, dim_localidad, poblacion_clean
)

dim_localidad_fecha.head()


dim_localidad_fecha lista. Shape: (576, 9)


,PK_LocalidadFecha,PK_Fecha,PK_Localidad,Ano,Mes,Localidad,Poblacion_Total,Poblacion_Hombres,Poblacion_Mujeres
0,1,202001,1,2020,1,Ciudad Bolivar,642989,315843,327146
1,2,202001,2,2020,1,Fontibon,383577,181305,202272
2,3,202001,3,2020,1,Kennedy,1046951,503223,543728
3,4,202001,4,2020,1,Puente Aranda,250855,122500,128355
4,5,202001,5,2020,1,San Cristobal,399766,193584,206182


## 5. Construcción de tablas de hechos

Con las dimensiones ya listas, construimos:

- `fact_mortalidad` → mortalidad mensual por localidad.
- `fact_momento`    → contaminación mensual por localidad (IBOCA + SISAIRE).

Ambas referencian `dim_fecha` y `dim_localidad` a través de sus PKs.


In [119]:
# 5.1 FACT_MORTALIDAD (solo localidades presentes en dim_localidad)

def construir_fact_mortalidad(
    medicina_mes_loc: pd.DataFrame,
    dim_fecha: pd.DataFrame,
    dim_localidad: pd.DataFrame,
) -> pd.DataFrame:
    if medicina_mes_loc.empty:
        print("⚠️ medicina_mes_loc está vacío. Devuelvo DataFrame vacío.")
        return pd.DataFrame()

    df = medicina_mes_loc.merge(
        dim_fecha[["PK_Fecha", "Ano", "Mes"]],
        on=["Ano", "Mes"],
        how="left"
    )

    df = df.merge(
        dim_localidad[["PK_Localidad", "Localidad"]],
        on="Localidad",
        how="left"
    )

    # Eliminar filas cuya localidad no esté en la intersección (PK_Localidad nulo)
    df = df.dropna(subset=["PK_Localidad"])

    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)
    df.insert(0, "PK_Mortalidad", df.index + 1)

    cols_muertes = [c for c in df.columns if c.startswith("Muertes_")]

    columnas = (
        ["PK_Mortalidad", "PK_Fecha", "PK_Localidad", "Ano", "Mes", "Localidad", "Fecha"]
        + cols_muertes
    )
    fact = df[columnas]

    print("fact_mortalidad lista. Shape:", fact.shape)
    return fact


fact_mortalidad = construir_fact_mortalidad(
    medicina_mes_loc, dim_fecha, dim_localidad
)
fact_mortalidad.head()


fact_mortalidad lista. Shape: (496, 12)


,PK_Mortalidad,PK_Fecha,PK_Localidad,Ano,Mes,Localidad,Fecha,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Muertes_CIE10_Agrupada,Muertes_CIE10_Basica
0,1,202001,1.0,2020,1,Ciudad Bolivar,2020-01-01,16,14,2,16,16
1,2,202001,2.0,2020,1,Fontibon,2020-01-01,4,2,2,4,4
2,3,202001,3.0,2020,1,Kennedy,2020-01-01,23,11,12,23,23
3,4,202001,4.0,2020,1,Puente Aranda,2020-01-01,9,2,7,9,9
4,5,202001,5.0,2020,1,San Cristobal,2020-01-01,11,5,6,11,11


In [120]:
# 5.2 FACT_MOMENTO (solo localidades presentes en dim_localidad)

def construir_fact_momento(
    iboca_mes_loc: pd.DataFrame,
    sisaire_mes_loc: pd.DataFrame,
    dim_fecha: pd.DataFrame,
    dim_localidad: pd.DataFrame,
    fact_mortalidad: pd.DataFrame,
) -> pd.DataFrame:
    if iboca_mes_loc.empty and sisaire_mes_loc.empty:
        print("⚠️ No hay datos de iboca_mes_loc ni sisaire_mes_loc.")
        return pd.DataFrame()

    df = pd.merge(
        iboca_mes_loc,
        sisaire_mes_loc,
        on=["Ano", "Mes", "Localidad", "Fecha"],
        how="outer"
    )

    df = df.merge(
        dim_fecha[["PK_Fecha", "Ano", "Mes"]],
        on=["Ano", "Mes"],
        how="left"
    )

    df = df.merge(
        dim_localidad[["PK_Localidad", "Localidad"]],
        on="Localidad",
        how="left"
    )

    # Filtrar a localidades presentes en dim_localidad
    df = df.dropna(subset=["PK_Localidad"])

    if not fact_mortalidad.empty:
        df = df.merge(
            fact_mortalidad[["PK_Mortalidad", "PK_Fecha", "PK_Localidad"]],
            on=["PK_Fecha", "PK_Localidad"],
            how="left"
        )
        df.rename(columns={"PK_Mortalidad": "FK_Mortalidad"}, inplace=True)
    else:
        df["FK_Mortalidad"] = None

    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)
    df.insert(0, "PK_Momento", df.index + 1)

    columnas = [
        "PK_Momento",
        "PK_Fecha",
        "PK_Localidad",
        "FK_Mortalidad",
        "Ano", "Mes", "Localidad", "Fecha",
        "Concentracion_Promedio",
        "NowCast_Promedio",
        "IBOCA_Promedio",
        "NO2_Promedio",
        "O3_Promedio",
        "SO2_Promedio",
        "CO_Promedio",
    ]
    columnas_finales = [c for c in columnas if c in df.columns]

    fact = df[columnas_finales]

    print("fact_momento lista. Shape:", fact.shape)
    return fact


fact_momento = construir_fact_momento(
    iboca_mes_loc,
    sisaire_mes_loc,
    dim_fecha,
    dim_localidad,
    fact_mortalidad,
)
fact_momento.head()


fact_momento lista. Shape: (564, 15)


,PK_Momento,PK_Fecha,PK_Localidad,FK_Mortalidad,Ano,Mes,Localidad,Fecha,Concentracion_Promedio,NowCast_Promedio,IBOCA_Promedio,NO2_Promedio,O3_Promedio,SO2_Promedio,CO_Promedio
0,1,202001,2.0,2.0,2020.0,1.0,Fontibon,2020-01-01,19.038043,19.359677,65.257769,NaN,NaN,NaN,NaN
1,2,202001,3.0,3.0,2020.0,1.0,Kennedy,2020-01-01,23.707713,23.670296,74.957742,33.802356,40.905034,4.382786,780.147598
2,3,202001,4.0,4.0,2020.0,1.0,Puente Aranda,2020-01-01,14.323288,14.688498,52.628796,31.752486,19.387378,4.396192,541.464885
3,4,202001,5.0,5.0,2020.0,1.0,San Cristobal,2020-01-01,11.977870,12.309655,45.722772,NaN,26.296755,NaN,NaN
4,5,202001,6.0,6.0,2020.0,1.0,Suba,2020-01-01,16.752368,16.484885,58.322901,NaN,28.271755,5.939822,NaN


## 6. Exportar tablas finales a CSV

Por último, guardamos **solo** las 5 tablas del modelo final:

1. `dim_fecha.csv`
2. `dim_localidad.csv`
3. `dim_localidad_fecha.csv`
4. `fact_mortalidad.csv`
5. `fact_momento.csv`


In [139]:
# ============================================================
# 🚫 ELIMINAR FECHAS/LOCALIDADES INCOMPLETAS EN fact_momento
# ============================================================

# 1. Detectar columnas que son indicadores ambientales
cols_indicadores = [
    c for c in fact_momento.columns
    if c not in ["PK_Momento", "PK_Fecha", "PK_Localidad", "Ano", "Mes", "Localidad", "Fecha"]
]

# 2. Filtrar SOLO filas donde todos los indicadores existan
fact_momento = fact_momento.dropna(subset=cols_indicadores, how="any")

# 3. Además eliminar fechas que no existan en TODAS las bases
fechas_validas = (
    set(fact_mortalidad["PK_Fecha"])
    & set(dim_localidad_fecha["PK_Fecha"])
)

fact_momento = fact_momento[fact_momento["PK_Fecha"].isin(fechas_validas)]

print("fact_momento filtrado. Shape:", fact_momento.shape)
fact_momento.head()


fact_momento filtrado. Shape: (343, 15)


,PK_Momento,PK_Fecha,PK_Localidad,FK_Mortalidad,Ano,Mes,Localidad,Fecha,Concentracion_Promedio,NowCast_Promedio,IBOCA_Promedio,NO2_Promedio,O3_Promedio,SO2_Promedio,CO_Promedio
1,2,202001,3.0,3.0,2020.0,1.0,Kennedy,2020-01-01,23.707713,23.670296,74.957742,33.802356,40.905034,4.382786,780.147598
2,3,202001,4.0,4.0,2020.0,1.0,Puente Aranda,2020-01-01,14.323288,14.688498,52.628796,31.752486,19.387378,4.396192,541.464885
7,8,202002,3.0,11.0,2020.0,2.0,Kennedy,2020-02-01,28.884560,28.773563,86.024842,43.100505,39.142326,7.161302,943.715260
8,9,202002,4.0,12.0,2020.0,2.0,Puente Aranda,2020-02-01,26.843705,26.665661,81.401135,45.157751,19.577957,4.781990,815.831258
13,14,202003,3.0,19.0,2020.0,3.0,Kennedy,2020-03-01,38.843962,38.988978,108.938522,35.435603,56.652750,5.204201,759.797273


In [141]:
# 🔥 FILTRO FINAL MUY EXPLÍCITO DE AÑOS INDESEADOS 🔥
# Pega esta celda AL FINAL del notebook, antes de los to_csv.

def quitar_anos_indeseados(df, anos_a_quitar, col_ano="Ano"):
    """
    Devuelve el DataFrame sin las filas cuyo año (col_ano) esté en anos_a_quitar.
    """
    if df is None or df.empty:
        return df
    return df[~df[col_ano].isin(anos_a_quitar)].copy()


# 👉 AÑOS QUE QUIERES QUITAR DE TODO (cámbialos si quieres más)
ANOS_A_QUITAR = [2025]

print("Años que voy a quitar explícitamente de todas las tablas:", ANOS_A_QUITAR)

# Aplicar el filtro a TODAS las tablas que tienen columna 'Ano'
dim_fecha_f = quitar_anos_indeseados(dim_fecha, ANOS_A_QUITAR, col_ano="Ano")
dim_localidad_fecha_f = quitar_anos_indeseados(dim_localidad_fecha, ANOS_A_QUITAR, col_ano="Ano")
fact_mortalidad_f = quitar_anos_indeseados(fact_mortalidad, ANOS_A_QUITAR, col_ano="Ano")
fact_momento_f = quitar_anos_indeseados(fact_momento, ANOS_A_QUITAR, col_ano="Ano")

print("Shapes después de quitar años indeseados:")
print("  dim_fecha:", dim_fecha_f.shape)
print("  dim_localidad_fecha:", dim_localidad_fecha_f.shape)
print("  fact_mortalidad:", fact_mortalidad_f.shape)
print("  fact_momento:", fact_momento_f.shape)

# ============================================================
# 📌 CREAR TABLA PUENTE DE TASAS fact_tasas_mortalidad_f
# ============================================================

# Subconjuntos de población y mortalidad ya filtrados
pob_f = dim_localidad_fecha_f[[
    "PK_Fecha",
    "PK_Localidad",
    "Poblacion_Total",
    "Poblacion_Hombres",
    "Poblacion_Mujeres"
]]

mort_f = fact_mortalidad_f[[
    "PK_Fecha",
    "PK_Localidad",
    "Muertes_Totales",
    "Muertes_Totales_Hombres",
    "Muertes_Totales_Mujeres"
]]

# Merge temporal SOLO para calcular tasas
tmp_f = mort_f.merge(
    pob_f,
    on=["PK_Fecha", "PK_Localidad"],
    how="inner"
)

# Crear tabla puente final
fact_tasas_mortalidad_f = pd.DataFrame()
fact_tasas_mortalidad_f["PK_Fecha"] = tmp_f["PK_Fecha"]
fact_tasas_mortalidad_f["PK_Localidad"] = tmp_f["PK_Localidad"]

fact_tasas_mortalidad_f["Tasa_Mortalidad_Total"] = (
    tmp_f["Muertes_Totales"] / tmp_f["Poblacion_Total"]
)

fact_tasas_mortalidad_f["Tasa_Mortalidad_Hombres"] = (
    tmp_f["Muertes_Totales_Hombres"] / tmp_f["Poblacion_Hombres"]
)

fact_tasas_mortalidad_f["Tasa_Mortalidad_Mujeres"] = (
    tmp_f["Muertes_Totales_Mujeres"] / tmp_f["Poblacion_Mujeres"]
)

# Tasas por 100k habitantes
factor = 100000

fact_tasas_mortalidad_f["Tasa_Total_100k"] = fact_tasas_mortalidad_f["Tasa_Mortalidad_Total"] * factor
fact_tasas_mortalidad_f["Tasa_Hombres_100k"] = fact_tasas_mortalidad_f["Tasa_Mortalidad_Hombres"] * factor
fact_tasas_mortalidad_f["Tasa_Mujeres_100k"] = fact_tasas_mortalidad_f["Tasa_Mortalidad_Mujeres"] * factor



Años que voy a quitar explícitamente de todas las tablas: [2025]
Shapes después de quitar años indeseados:
  dim_fecha: (60, 4)
  dim_localidad_fecha: (480, 9)
  fact_mortalidad: (480, 12)
  fact_momento: (335, 15)


In [142]:
# ============================================================
# 💾 EXPORTAR TABLAS DEFINITIVAS FILTRADAS
# ============================================================

dim_fecha_f.to_csv(RUTA_SALIDA / "dim_fecha.csv", index=False)
dim_localidad.to_csv(RUTA_SALIDA / "dim_localidad.csv", index=False)
dim_localidad_fecha_f.to_csv(RUTA_SALIDA / "dim_localidad_fecha.csv", index=False)
fact_mortalidad_f.to_csv(RUTA_SALIDA / "fact_mortalidad.csv", index=False)
fact_momento_f.to_csv(RUTA_SALIDA / "fact_momento.csv", index=False)

# 🌟 NUEVA TABLA PUENTE
fact_tasas_mortalidad_f.to_csv(RUTA_SALIDA / "fact_tasas_mortalidad.csv", index=False)

print("🎉 Archivos exportados correctamente en", RUTA_SALIDA)


🎉 Archivos exportados correctamente en Output_ETL
